**14 - Responsible AI Foundations**

**1. Notebook Objective**


This notebook introduces Responsible AI (RAI) concepts for the Product
Defect Computer Vision project.

The goal is to understand how to evaluate whether an AI system can be
trusted and used appropriately, beyond simply measuring model accuracy.

Topics include:

1. What Responsible AI means.
2. Explainable AI versus Responsible AI.
3. Reliability and safety.
4. Fairness and performance across relevant groups.
5. Data and evaluation integrity.
6. Confidence and uncertainty.
7. Transparency and model limitations.
8. Human oversight and accountability.
9. Responsible AI across the complete ML lifecycle.
10. How these concepts apply to the product-defect classifier.

**2. What is Responsible AI?**

We have spent most of this project asking:

Can we build a model that predicts: DEFECT or OK?

Responsible AI introduces a broader question:

Can this AI system be trusted and used appropriately for its intended purpose?

Notice I said AI system, not simply model.

A model can have excellent accuracy while the overall system still has serious problems.

For example, imagine:

Product Defect Classifier

Test accuracy = 99.5%

That sounds excellent.

But then we discover:

Test data contains leakage
              OR

model fails under different lighting
              OR

model confidently misses certain defects
              OR

operators interpret Grad-CAM as exact
defect localization
              OR

production camera changes and
performance deteriorates

The accuracy number alone doesn't answer those questions.

That is where RAI comes in.

**3. XAI versus RAI**

**Explainable AI **

XAI asks questions such as: 

- Why did the model predict OK?
- Which image regions influenced DEFECT?
- Is the model focusing on unexpected areas?

Grad-CAM was one tool for investigating these questions.

**⚖️ Responsible AI**

RAI asks broader questions:

- Can we trust the data?
- Can we trust the evaluation?
- What kinds of mistakes does the model make?
- What happens when the model is wrong?
- Does it behave reliably under realistic conditions?
- Are users told what the model can and cannot do?
- Should a human review certain predictions?
- How will we detect problems after deployment?

Therefore:

``` text


             RESPONSIBLE AI
                    │
     ┌──────────────┼───────────────┐
     ↓              ↓               ↓
Data integrity   Reliability    Transparency
     │              │               │
     ├──────────────┼───────────────┤
     ↓              ↓               ↓
Fairness          XAI          Human oversight
     │                              │
     └──────────────┬───────────────┘
                    ↓
              Appropriate use

```
XAI can contribute evidence to RAI, but: XAI is not the same thing as Responsible AI.

##### 4. Responsible AI starts before training

``` text

DATA
 ↓
RAI checks

PREPROCESSING
 ↓
RAI checks

TRAINING
 ↓
RAI checks

EVALUATION
 ↓
RAI checks

DEPLOYMENT
 ↓
RAI checks

MONITORING
 ↓
RAI checks

```

**5. Stage 1 — Data integrity**

- Where did the data come from?
- Who created the labels?
- What exactly does "def_front" mean?
- Are images corrupted?
- Are classes imbalanced?
- Are duplicate images present?
- Could the same physical product appear
- in train and test?
- Does the dataset represent the environment
- where the model will actually be used?

This is partly normal EDA/data quality and partly Responsible AI.

The categories overlap.

The important thing isn't whether we label a check "EDA" or "RAI." The important thing is whether we perform it.

**6. Our project already found a real example**

Remember our dataset:

- Development pool: 6,633 images
- Test:               715 images

Our SHA-256 audit found:

64 exact byte-identical images
shared between the development pool
and the supplied test folder

Approximately:

64 / 715 ≈ 8.95%

This is an evaluation-integrity issue.

Why?

A test set is supposed to provide evidence about performance on data that did not participate in model development.

Conceptually:

Development data

``` text

train
  +
validation

       X

independent test data

But exact duplicates create:

Development
     ↘
      same image
     ↗
Test

```

Therefore the supplied test set is not completely independent of the development data.

**⚖️ RAI lesson**

A metric can be mathematically calculated correctly while the evaluation design itself is flawed.

That's an important distinction:

Correct metric calculation
        ≠
trustworthy evaluation

##### 7. But we investigated rather than panicked

We then removed the exact development/test duplicates from the analysis.

ResNet18 changed approximately:

- Original test: 99.58%
- Non-duplicate subset: 99.54%

And all three ResNet false negatives were in the non-duplicate subset.

So our conclusion was not:

``` text

Duplicates exist
      ↓
model is useless

Instead:

Duplicates exist
      ↓
evaluation integrity concern
      ↓
measure impact
      ↓
performance remains very similar
      ↓
document limitation

```

That is good Responsible AI reasoning.

RAI isn't about automatically rejecting a model whenever a problem is discovered.

It's about identifying, measuring, understanding, mitigating where appropriate, and documenting risks.

##### 8. Stage 2 — Preprocessing integrity

Now consider image preprocessing.

We used augmentations such as:

- RandomHorizontalFlip
- RandomRotation
- RandomResizedCrop

Why did we change the pretrained crop configuration?

The default RandomResizedCrop could potentially remove a large portion of an industrial defect while retaining the image's original defect label.

That could create:

``` text

Original image:
DEFECT visible

        ↓ aggressive crop

Defect disappears

        ↓

Label still says:
DEFECT

```
Now we've accidentally created noisy or misleading training data.

That's why we made the augmentation more conservative.

**⚖️ RAI principle**

Training transformations should be plausible for the intended production environment and preserve the meaning of the label.

This applies beyond computer vision.

For NLP:

- text augmentation
- must preserve intended meaning

For tabular data:

- synthetic transformations
- must preserve realistic relationships

##### 9. Stage 3 — Evaluation beyond accuracy

Consider our custom CNN.

Test accuracy: 83.22%

We could stop there.

But we examined the confusion matrix:

                Predicted
              DEFECT    OK

Actual DEFECT   385     68

Actual OK        52    210

The number: 68 is particularly important.

These are:

actual defect
      ↓
predicted OK

False negatives.

In a manufacturing quality-control scenario, those could mean defective products being accepted.

So:

Accuracy = 83.22% doesn't tell the whole story.

We need:

- per-class precision
- per-class recall
- confusion matrix
- false negatives
- false positives
- failure cases

**⚖️ RAI principle**

Evaluate errors according to their real-world consequences, not only their mathematical frequency.

##### 10. False negatives versus false positives

Let's connect this to the business problem.

**False negative**

- Actual: DEFECT
- Model: OK

Possible consequence: defective product passes inspection


**False positive**

- Actual: OK
- Model: DEFECT

Possible consequence: good product gets rejected / manually inspected

Both matter, but their costs may differ.

In many quality-control contexts: missed defect

could be more serious than: unnecessary inspection

But that isn't something the ML engineer should silently assume. The appropriate cost tradeoff should come from the business, safety, quality, and domain requirements.

That's also Responsible AI.

##### 11. Our ResNet example

Our ResNet had:

- Actual defect images = 453
- Correctly detected = 450
- Missed defects = 3

Defect recall was approximately: 99.34% Very strong.

But Responsible AI does not say: 99.34%  -> done

Instead:

- What are those 3 missed defects?
- Are they similar?
- Are they subtle?
- Does the model fail under particular conditions?
- Are these failure modes acceptable?
- Should low-confidence cases receive human review?
- That's why Notebook 13 analyzed those exact images.

##### 12. Stage 4 — Confidence is not trust

This is one of the most important concepts we've encountered.

One of our false negatives was approximately:

- Actual: DEFECT
- Predicted: OK
- OK probability:98.46%

So:

HIGH CONFIDENCE
      +
WRONG

is absolutely possible.

The model is effectively saying:

- Among my available classes, I strongly prefer OK.
- It is not saying: There is a 98.46% guarantee that this product is truly OK.

Those statements are different.

##### 13. Confidence versus calibration

This introduces another concept: calibration.

Imagine 100 predictions where a model reports approximately: confidence ≈ 90%

A well-calibrated model would ideally be correct roughly: ~90 of those 100 times

Conceptually:

``` text

Predicted confidence
       ↓
Does observed correctness
match that confidence?

```

- A model can have: high accuracy
- while still being: poorly calibrated

These are different properties.

We'll keep this conceptual here. Notebook 15 can assess confidence and failure behavior using our actual model outputs.

##### 14. Stage 5 — Reliability

Reliability asks: Does the model continue to behave appropriately under conditions it is expected to encounter?

Our images may vary by:

- lighting
- brightness
- camera
- position
- rotation
- surface condition
- background
- manufacturing batch
- equipment

Suppose our model performs extremely well on today's images: 99.5%

Then the factory changes its camera.

Suddenly:

- brightness changes
- contrast changes
- resolution changes

and accuracy drops.

The original test accuracy did not tell us this would happen.

This is why deployment introduces another issue:

**Data drift / distribution shift**

``` text

Training environment
        ↓
Camera A
Lighting A
Process A

        versus

Production later
        ↓
Camera B
Lighting B
Process B

```

The model may receive data different from what it learned.

##### 15. This connects directly to our Grad-CAM discussion

In Notebook 13 we asked:

What if Grad-CAM repeatedly highlights background, lighting, or borders?

Grad-CAM alone cannot prove shortcut learning.

But it can generate a hypothesis:

``` text

Grad-CAM
highlights suspicious background region
        ↓
hypothesis
        ↓
change / mask background
        ↓
rerun prediction
        ↓
does prediction change substantially?

```

Now we're moving toward a reliability experiment.

That's the bridge:

🔍 XAI
"What looks suspicious?"

        ↓

⚖️ RAI
"Does that suspicious behavior create
a reliability risk?"

##### 16. Stage 6 — Fairness

Fairness is sometimes misunderstood as applying only to demographic attributes.

The broader idea is: Does the system perform appropriately across relevant groups or conditions?

For a hiring model, relevant groups might include demographic populations, subject to legal and ethical requirements.

For our industrial vision model, relevant slices might instead be:

- different defect types
- different casting batches
- different machines
- different cameras
- different lighting conditions
- different product variants

Suppose overall accuracy is: 99%

but:

Machine A images → 99.8%
Machine B images → 81%

Overall accuracy could hide an important reliability disparity.

Our current dataset has only:

def_front
ok_front

and doesn't appear to provide the rich operational metadata we'd need for those analyses.

That's itself a limitation worth documenting.

##### 17. Stage 7 — Transparency

Transparency means users should understand important facts about the system.

For our classifier, that includes things like:

- Intended use: classify casting front images as def_front or ok_front
- Not intended for: precise defect localization

- Training/evaluation limitations:
    - dataset-specific conditions
    - exact development/test duplicates discovered
    - limited operational metadata

- Known model behavior:
    - false negatives exist
    - confidence can be high when wrong

- XAI limitation:
    - Grad-CAM shows broad influential regions,
    - not verified defect locations

Transparency is not simply: "Here's the neural network architecture."

It also means communicating what the system can and cannot support.

##### 18. Stage 8 — Human oversight

Imagine deploying this classifier in a factory.

A dangerous design could be:

``` text

Image
 ↓
AI says OK
 ↓
automatically ship product

```

without considering error consequences.

A different design might use:

``` text

Image
 ↓
Model prediction
 ↓
confidence + business rules
 ↓
┌─────────────────────┐
│                     │
high-confidence      uncertain /
routine case         high-risk case
│                     │
↓                     ↓
normal process       human inspection

```
Exactly where those thresholds belong is a business/domain decision that should be validated rather than guessed.

⚖️ Human oversight principle

AI can support a decision without necessarily being the sole decision-maker.

##### 19. Accountability

Another RAI question is: Who is responsible when something goes wrong?

Not: "The AI decided it."

A production system should establish responsibilities around:

- Who approved the model?
- Who monitors it?
- Who investigates failures?
- Who can stop deployment?
- Who reviews escalated cases?
- How are model versions tracked?
- How are predictions audited?

This connects directly to technologies we're already using:

- MLflow
- model versions
- registered models
- deployment versions
- logging
- monitoring

MLOps and Responsible AI overlap considerably here.

###### 20. RAI across our complete project

``` text
DATA
│
├─ source/provenance
├─ labels
├─ class distribution
├─ duplicates
├─ leakage
└─ representativeness
        ↓

PREPROCESSING
│
├─ resize
├─ normalization
├─ augmentation realism
└─ label preservation
        ↓

TRAINING
│
├─ reproducibility
├─ model selection
└─ validation discipline
        ↓

EVALUATION
│
├─ accuracy
├─ precision
├─ recall
├─ confusion matrix
├─ false negatives
├─ false positives
├─ confidence
└─ failure analysis
        ↓

🔍 XAI
│
├─ Grad-CAM
├─ correct predictions
├─ incorrect predictions
└─ suspicious regions
        ↓

⚖️ RAI
│
├─ reliability
├─ evaluation integrity
├─ fairness / relevant slices
├─ transparency
├─ limitations
├─ human oversight
└─ accountability
        ↓

DEPLOYMENT
│
├─ monitoring
├─ drift
├─ model versioning
├─ auditability
└─ escalation
```

##### 21. A useful distinction: Model quality vs system responsibility

MODEL QUALITY
─────────────
- How accurate is it?
- What is recall?
- What is F1?
- How many errors?


RESPONSIBLE AI
──────────────
- Can we trust the evaluation?
- Where does it fail?
- Who is affected by failures?
- What happens when it is wrong?
- Are users informed of limitations?
- Does it remain reliable in production?
- When should humans intervene?
- Who monitors and owns the system?

Model quality is part of Responsible AI, but Responsible AI is broader.

##### 22. Key Learnings


1. Responsible AI applies across the entire AI lifecycle, not only after
   model training.

2. Responsible AI evaluates the broader AI system, not only the model's
   accuracy.

3. Explainable AI is one component that can provide evidence for a broader
   Responsible AI assessment.

4. Data quality, provenance, duplicates, leakage, and split independence
   affect whether model evaluation can be trusted.

5. A metric can be calculated correctly while the evaluation design itself
   is unreliable.

6. Training augmentation should be realistic and label-preserving.

7. Overall accuracy can hide important class-specific failure modes.

8. False negatives and false positives should be evaluated according to
   their real-world consequences.

9. High prediction confidence does not guarantee correctness.

10. Confidence and calibration are different concepts.

11. Reliability includes behavior under realistic changes such as lighting,
    camera conditions, equipment, and data drift.

12. Fairness should be evaluated across relevant populations or operational
    groups when the necessary data is available.

13. Transparency requires clearly communicating intended use, limitations,
    known failure modes, and the meaning of model explanations.

14. Grad-CAM provides broad class-influence information but should not be
    presented as verified defect localization.

15. Human oversight may be necessary when AI errors have meaningful
    consequences.

16. Accountability requires ownership, monitoring, versioning, auditing,
    and escalation processes.

17. Responsible AI does not mean that a model must be perfect. It means
    risks and limitations should be identified, evaluated, mitigated where
    appropriate, monitored, and communicated.

##### 23. Notebook Conclusion


- Responsible AI extends model development beyond predictive performance.

- For the Product Defect Computer Vision project, several Responsible AI
considerations have already emerged from the development process.

- The dataset contains moderate class imbalance and an exact-duplicate audit
identified overlap between the development pool and the supplied test data,
demonstrating why evaluation integrity must be assessed rather than assumed.

- Model evaluation showed that overall accuracy alone is insufficient.
False-negative defects are particularly important because they represent
defective products classified as OK.

- The fine-tuned ResNet18 achieved strong classification performance but still
produced false negatives, including a high-confidence incorrect prediction.
This demonstrates that model confidence should not be interpreted as a
guarantee of correctness.

- Grad-CAM provided additional insight into broad regions that influenced
individual class scores, but it did not provide verified defect
localization or prove causal model reliance on highlighted regions.

- Responsible deployment therefore requires consideration of data integrity,
failure modes, reliability under realistic conditions, transparency,
appropriate human oversight, monitoring, and accountability.

- The next notebook will move from these foundations to a practical
Responsible AI assessment of the trained product-defect classifier using
the evidence collected throughout this project.